# Procurement Agent

Agent to decide which product categories suit the aesthetic, generates one Google Shopping query per category, and fetches real purchasable products via SerpAPI based on the stylist agent's style profile

Uses `Qwen/Qwen2.5-7B-Instruct` via the **HuggingFace Inference API** — no local model download or GPU required.

> **Before running:** Create a `.env` file in the project root with `HF_TOKEN` and `SERPAPI_API_KEY`.

In [14]:
import os
from pathlib import Path
from getpass import getpass

# Load from .env file if present (place it at the project root)
try:
    from dotenv import load_dotenv
    # Walk up from this notebook to find the project root .env
    env_path = Path('../../.env').resolve()
    if env_path.exists():
        load_dotenv(dotenv_path=env_path)
        print(f'Loaded .env from {env_path}')
    else:
        load_dotenv()  # fallback: search CWD and parents
        print('Attempted to load .env from working directory.')
except ImportError:
    pass

# Fall back to manual entry if keys are still missing
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('Enter your HuggingFace token (hf_...): ')
if not os.environ.get('SERPAPI_API_KEY'):
    os.environ['SERPAPI_API_KEY'] = getpass('Enter your SerpAPI key: ')

print('API keys ready.')

Loaded .env from /Users/raras/Library/CloudStorage/GoogleDrive-raras.pramudita@berkeley.edu/My Drive/02 Spring 2026/INFO 290 Fundamentals Gen AI/VisionCart/.env
API keys ready.


In [8]:
# ── tools/api.py (inlined) ──────────────────────────────────────────────────
from __future__ import annotations

import re
from typing import Any, Dict, List, Optional, Tuple
from urllib.parse import quote

import requests

Money = Tuple[Optional[float], Optional[str]]

_STOP_WORDS = {
    "a", "an", "the", "and", "or", "for", "of", "in", "on", "with",
    "to", "by", "at", "is", "it", "its", "as", "be", "set", "pack",
    "pcs", "pc", "piece", "pieces", "lot", "new", "sale", "free",
}


def _extract_title_tags(title: str) -> List[str]:
    tokens = re.split(r"[\-,|/&+()[\]{}]+", title.lower())
    tags = []
    for token in tokens:
        token = token.strip()
        if token and token not in _STOP_WORDS and not token.isdigit():
            tags.append(token)
    return tags


def _parse_price_to_money(price: Any) -> Money:
    if price is None:
        return (None, None)
    if isinstance(price, (int, float)):
        return (float(price), None)
    if not isinstance(price, str):
        return (None, None)
    s = price.strip()
    if not s:
        return (None, None)
    currency_map = {
        "$": "USD", "£": "GBP", "€": "EUR", "¥": "JPY",
        "₹": "INR", "₩": "KRW", "₫": "VND", "₺": "TRY",
        "R$": "BRL", "C$": "CAD", "A$": "AUD",
    }
    currency = None
    for sym, code in sorted(currency_map.items(), key=lambda x: -len(x[0])):
        if s.startswith(sym):
            currency = code
            s = s[len(sym):].strip()
            break
    m = re.search(r"(\d[\d,]*\.?\d*)", s)
    if not m:
        return (None, currency)
    try:
        value = float(m.group(1).replace(",", ""))
    except ValueError:
        return (None, currency)
    return (value, currency)


def serpapi_google_shopping_search(
    *,
    api_key: str,
    query: str,
    num: int = 10,
    country: str = "us",
    language: str = "en",
    timeout_s: int = 30,
) -> List[Dict[str, Any]]:
    if not api_key:
        raise ValueError("Missing SerpAPI api_key")
    if not query or not query.strip():
        return []
    params = {
        "engine": "google_shopping",
        "q": query,
        "api_key": api_key,
        "gl": country,
        "hl": language,
    }
    resp = requests.get("https://serpapi.com/search.json", params=params, timeout=timeout_s)
    resp.raise_for_status()
    data = resp.json()
    items = data.get("shopping_results") or []
    out: List[Dict[str, Any]] = []
    for it in items[:max(0, num)]:
        price_val, currency = _parse_price_to_money(it.get("price"))
        tags = _extract_title_tags(it.get("title") or it.get("name") or "")
        out.append({
            "image_url": it.get("thumbnail") or it.get("image"),
            "product_name": it.get("title") or it.get("name") or "",
            "price": price_val,
            "currency": currency,
            "product_url": quote(it.get("link") or it.get("product_link") or "", safe=":/?=&#%+@"),
            "source": it.get("source"),
            "tags": tags,
            "raw": it,
        })
    return [p for p in out if p.get("product_name") or p.get("product_url")]

In [9]:
import os
from huggingface_hub import InferenceClient

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'

client = InferenceClient(
    model=MODEL_ID,
    token=os.environ['HF_TOKEN'],
)
print(f'InferenceClient ready — model: {MODEL_ID}')

InferenceClient ready — model: Qwen/Qwen2.5-7B-Instruct


In [10]:
# ── procurement.py (HuggingFace Inference API) ──────────────────────────────
from __future__ import annotations

import json
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional


@dataclass(frozen=True)
class StylistOutput:
    style_profile: str
    aesthetic: List[str]
    colors: List[str]
    materials: List[str]
    budget_max: Optional[float] = None
    budget_currency: Optional[str] = None

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


def _build_query_prompt(
    style: StylistOutput,
    num_queries: int,
    critic_feedback: Optional[str],
) -> str:
    budget_line = (
        f"Budget: up to {style.budget_max} {style.budget_currency or 'USD'}"
        if style.budget_max
        else "Budget: no hard limit"
    )
    feedback_section = (
        f"\nThe previous search was evaluated and had these issues — address them in your new queries:\n{critic_feedback}\n"
        if critic_feedback
        else ""
    )
    return f"""You are a search query specialist for an aesthetic shopping platform.

Given a style profile, decide which product categories a shopper would want to buy to achieve this aesthetic, then generate one specific Google Shopping search query per category.

Style profile: {style.style_profile}
Colors: {", ".join(style.colors)}
Materials: {", ".join(style.materials)}
Aesthetics: {", ".join(style.aesthetic)}
{budget_line}
{feedback_section}
Rules:
- Generate exactly {num_queries} queries covering different shoppable product categories that suit this aesthetic.
- Each query must be specific and aesthetic-forward — include material, mood, or style era context.
- Avoid generic terms (e.g. "home decor" alone is too vague).
- Do NOT repeat a query angle from a previous run if critic feedback is provided.

Return a JSON array of strings and nothing else. No explanation, no markdown.
Example: ["handmade terracotta ceramic planter cottagecore", "rattan outdoor lantern boho warm patio"]"""


def build_queries(
    style: StylistOutput,
    num_queries: int,
    critic_feedback: Optional[str] = None,
) -> List[str]:
    """
    Calls the HuggingFace Inference API to generate shopping search queries
    from the style profile. No local model or GPU required.
    """
    print(f"[procurement] Calling HF Inference API (model: {MODEL_ID})...")
    if critic_feedback:
        print(f"[procurement] Retry pass — critic feedback:\n  {critic_feedback}")

    prompt = _build_query_prompt(style, num_queries, critic_feedback)
    response = client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=512,
    )
    raw = response.choices[0].message.content.strip()
    print(f"[procurement] Query generation complete. Raw response: {raw}")

    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    raw = raw.strip()

    parsed = json.loads(raw)
    # Model may return {"queries": [...]} or [{"query": ...}] instead of ["...", ...]
    if isinstance(parsed, dict):
        parsed = parsed.get("queries", [])
    if parsed and isinstance(parsed[0], dict):
        parsed = [item.get("query") or item.get("text") or "" for item in parsed]
    queries = [q for q in parsed if isinstance(q, str) and q.strip()]
    if not queries:
        raise RuntimeError(f"LLM returned an unexpected format for queries: {raw!r}")

    return [q.strip() for q in queries if q.strip()]


def _shape_products(items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    return [
        {
            "image_url": it.get("image_url", ""),
            "product_name": it.get("product_name", ""),
            "price": it.get("price"),
            "link": it.get("product_url", ""),
            "tags": it.get("tags") or [],
        }
        for it in items
    ]


def _resolve_style(state: Dict[str, Any]) -> StylistOutput:
    stylist_dict = state.get("stylist_output")
    if not stylist_dict:
        raise ValueError("stylist_output is required in state")
    return StylistOutput(
        style_profile=stylist_dict.get("style_profile", ""),
        aesthetic=list(stylist_dict.get("aesthetic") or []),
        colors=list(stylist_dict.get("colors") or []),
        materials=list(stylist_dict.get("materials") or []),
        budget_max=stylist_dict.get("budget_max"),
        budget_currency=stylist_dict.get("budget_currency"),
    )


def run(state: Dict[str, Any]) -> str:
    """
    Procurement agent entry point.

    state keys:
      - stylist_output (dict): style_profile, aesthetic, colors, materials, budget_max, budget_currency
      - results_per_query (int, optional): SerpAPI results per query (default 20)
      - num_queries (int, optional): number of search queries (default 5)
      - critic_feedback (str, optional): retry guidance from the critic agent

    Returns JSON string with procurement_queries, procurement_products, style_profile.
    """
    api_key = os.environ.get("SERPAPI_API_KEY")
    results_per_query = int(state.get("results_per_query") or 20)
    num_queries = int(state.get("num_queries") or 5)
    critic_feedback: Optional[str] = state.get("critic_feedback")

    style = _resolve_style(state)
    queries = build_queries(style, num_queries=num_queries, critic_feedback=critic_feedback)

    seen_products: set = set()
    all_products: List[Dict[str, Any]] = []

    print(f"[procurement] Fetching products for {len(queries)} queries (up to {results_per_query} results each)...")
    for i, q in enumerate(queries, 1):
        print(f"[procurement]   [{i}/{len(queries)}] Querying SerpAPI: '{q}'")
        before = len(all_products)
        for it in serpapi_google_shopping_search(api_key=api_key, query=q, num=results_per_query):
            key = (
                (it.get("product_url") or "").strip(),
                (it.get("product_name") or "").strip().lower(),
            )
            if key not in seen_products:
                seen_products.add(key)
                all_products.append(it)
        print(f"[procurement]          → {len(all_products) - before} new products (pool total: {len(all_products)})")

    print(f"[procurement] Done. Total candidate pool: {len(all_products)} products.")
    result = {
        "procurement_queries": queries,
        "procurement_products": _shape_products(all_products),
        "style_profile": style.style_profile,
    }
    return json.dumps(result, indent=2)

## Run the Procurement Agent

Edit the `state` dict below to match your stylist output, then run the cell.

In [11]:
state = {
    "stylist_output": {
        "style_profile": "Cozy cottagecore living space with warm earthy tones and natural textures",
        "aesthetic": ["cottagecore", "rustic", "vintage"],
        "colors": ["sage green", "terracotta", "cream", "warm brown"],
        "materials": ["rattan", "linen", "ceramic", "wood"],
        "budget_max": 150.0,
        "budget_currency": "USD",
    },
    "num_queries": 5,
    "results_per_query": 10,
}

output_json = run(state)
output = json.loads(output_json)

[procurement] Calling HF Inference API (model: Qwen/Qwen2.5-7B-Instruct)...
[procurement] Query generation complete. Raw response: ["sage green linen throw vintage cottagecore", "terracotta wood coffee table rustic living room", "warm brown rattan side chair cottagecore decor", "creamy linen curtains cottagecore window dressing", "earthy toned ceramic vase rustic home decor"]
[procurement] Fetching products for 5 queries (up to 10 results each)...
[procurement]   [1/5] Querying SerpAPI: 'sage green linen throw vintage cottagecore'
[procurement]          → 10 new products (pool total: 10)
[procurement]   [2/5] Querying SerpAPI: 'terracotta wood coffee table rustic living room'
[procurement]          → 10 new products (pool total: 20)
[procurement]   [3/5] Querying SerpAPI: 'warm brown rattan side chair cottagecore decor'
[procurement]          → 10 new products (pool total: 30)
[procurement]   [4/5] Querying SerpAPI: 'creamy linen curtains cottagecore window dressing'
[procurement]     

In [13]:
# Preview results
print("Queries generated:")
for q in output["procurement_queries"]:
    print(f"  • {q}")

print(f"\nTotal products fetched: {len(output['procurement_products'])}")
print("\nFirst 5 products:")
for p in output["procurement_products"][:5]:
    print(f"  [{p['price']}] {p['product_name']}")
    print(f"         {p['link']}")

Queries generated:
  • sage green linen throw vintage cottagecore
  • terracotta wood coffee table rustic living room
  • warm brown rattan side chair cottagecore decor
  • creamy linen curtains cottagecore window dressing
  • earthy toned ceramic vase rustic home decor

Total products fetched: 50

First 5 products:
  [100.0] Boho Woven Throw Blanket Sage Green Bedding Cottagecore Decor Aesthetic Tapestry Sage GREEN Woven Blanket MOTH Woven BLANKET Gifts for Her
         https://www.google.com/search?ibp=oshop&q=sage%20green%20linen%20throw%20vintage%20cottagecore&prds=productid:7277474839347512471%2CheadlineOfferDocid:7277474839347512471%2CimageDocid:5727896521089475527%2Crds:LO_7277474839347512471%7CPROD_LO_7277474839347512471%2Cpvt:hg&hl=en&gl=us&udm=28
  [35.0] Threshold Scalloped Edge Throw
         https://www.google.com/search?ibp=oshop&q=sage%20green%20linen%20throw%20vintage%20cottagecore&prds=catalogid:16805687174314453276%2CheadlineOfferDocid:11097540968439453004%2CimageDoci